1\. Environment Setup
---------------------

The authors utilized a Linux-based server with NVIDIA RTX GPUs. For this replication, we use **Detectron2**, a standard library for object detection research.

**Kaggle Note:** We install Detectron2 from source to ensure compatibility with Kaggle's pre-installed PyTorch version. We also ensure pyyaml is pinned to prevent dependency conflicts.

In [2]:
# Installation (Uncomment if needed)
!pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu111/torch1.9/index.html

import os
import copy
import torch
import numpy as np
from datetime import datetime
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, default_setup
from detectron2.data import DatasetCatalog, MetadataCatalog, build_detection_train_loader
from detectron2.data import transforms as T
from detectron2.data import detection_utils as utils
from detectron2 import model_zoo

print(f"Using Torch version: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

Looking in links: https://dl.fbaipublicfiles.com/detectron2/wheels/cu111/torch1.9/index.html
ERROR: Could not find a version that satisfies the requirement detectron2 (from versions: none)
ERROR: No matching distribution found for detectron2


ModuleNotFoundError: No module named 'detectron2'

2\. Dataset Registration
------------------------

As per **Section 3.1**, the dataset consists of 5,900 images of paprika plants. We follow the paper's split:

*   **70% Training**
    
*   **10% Validation**
    
*   **20% Testing**
    

The DDL unit focuses on 6 abnormality categories found in the Paprika dataset.

In [ ]:
# 6 Classes from the paper (Section 3.1.2)
CLASS_NAMES = [
    "Blossom end rot", 
    "Powdery mildew", 
    "Gray mold", 
    "Cercospora", 
    "Snails and slugs", 
    "Spider mite"
]

# Path to your dataset in Kaggle input (Adjust this path to match your uploaded dataset)
DATASET_ROOT = "/kaggle/input/paprika-plant-disease-dataset" 

def register_paprika_datasets():
    """
    Registers the Train, Val, and Test sets.
    Assumes COCO JSON format: annotations/train.json, images/train/ etc.
    """
    from detectron2.data.datasets import register_coco_instances
    
    # Example structure - Adjust based on your actual Kaggle dataset layout
    # If you haven't uploaded the dataset yet, this will just register the names
    for split in ["train", "val", "test"]:
        name = f"paprika_{split}"
        
        # Paths (Placeholder structure)
        json_file = os.path.join(DATASET_ROOT, f"annotations_{split}.json")
        image_dir = os.path.join(DATASET_ROOT, "images")
        
        # Check if file exists to avoid errors if dataset isn't loaded yet
        if os.path.exists(json_file):
            register_coco_instances(name, {}, json_file, image_dir)
            
            # Set metadata
            MetadataCatalog.get(name).set(thing_classes=CLASS_NAMES)
            print(f"Registered: {name}")
        else:
            print(f"Warning: Could not find {json_file}. Please upload dataset.")

register_paprika_datasets()

# Verification: Visualize a sample (Only works if dataset is loaded)
if "paprika_train" in DatasetCatalog.list():
    dataset_dicts = DatasetCatalog.get("paprika_train")
    # Visualize 1 random sample
    import random
    d = random.sample(dataset_dicts, 1)[0]
    img = utils.read_image(d["file_name"], format="BGR")
    visualizer = Visualizer(img[:, :, ::-1], metadata=MetadataCatalog.get("paprika_train"), scale=0.5)
    out = visualizer.draw_dataset_dict(d)
    plt.figure(figsize=(10, 10))
    plt.imshow(out.get_image()[:, :, ::-1])
    plt.show()
